In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
PAD, UNK = 0, 1
def build_vocab(texts, min_freq=2, max_size=30000):
    cnt = Counter()
    for t in texts:
        cnt.update(re.findall(r"[a-zA-Z']+|\d+", t.lower()))
    most_common = [w for w, c in cnt.most_common(max_size) if c >= min_freq]
    vocab = {'<pad>': PAD, '<unk>': UNK}
    for w in most_common:
        vocab[w] = len(vocab)
    return vocab

def encode_text(text, vocab, max_len=80):
    toks = re.findall(r"[a-zA-Z']+|\d+", text.lower())[:max_len]
    ids = [vocab.get(t, UNK) for t in toks]
    if len(ids) < max_len:
        ids = ids + [PAD] * (max_len - len(ids))
    return np.array(ids, dtype=np.int64), min(len(toks), max_len)


all_text = [train['prompt_n'].tolist(), test['prompt_n'].tolist()]
for o in OPTS:
    all_text.append(train[f'{o}_n'].tolist())
    all_text.append(test[f'{o}_n'].tolist())
VOCAB = build_vocab([t for grp in all_text for t in grp])
print(f'LSTM vocab size: {len(VOCAB)}')

In [ ]:
class LSTMDataset(Dataset):
    def __init__(self, df, vocab, max_len=80, shuffle_opts=True, has_label=True, seed=0):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab
        self.max_len = max_len
        self.shuffle = shuffle_opts
        self.has_label = has_label
        self.rng = np.random.default_rng(seed)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        q_ids, q_len = encode_text(r['prompt_n'], self.vocab, self.max_len)
        opt_data = [(encode_text(r[f'{o}_n'], self.vocab, self.max_len)) for o in OPTS]
        opt_ids = np.stack([x[0] for x in opt_data])
        opt_lens = np.array([x[1] for x in opt_data], dtype=np.int64)
        opt_lens = np.maximum(opt_lens, 1)  

        correct = OPTS.index(r['answer']) if self.has_label else -1
        if self.shuffle:
            perm = self.rng.permutation(5)
            opt_ids = opt_ids[perm]
            opt_lens = opt_lens[perm]
            if self.has_label:
                correct = int(np.where(perm == correct)[0][0])

        return dict(
            q=torch.from_numpy(q_ids),
            q_len=torch.tensor(max(q_len, 1)),
            opts=torch.from_numpy(opt_ids),
            opt_lens=torch.from_numpy(opt_lens),
            correct=torch.tensor(correct, dtype=torch.long),
            id=int(r['id']),
        )

In [ ]:
class SiameseBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=200, hidden=256, dropout=0.3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(embed_dim, hidden, batch_first=True,
                             bidirectional=True, num_layers=1)
        self.dropout = nn.Dropout(dropout)
      
        self.head = nn.Sequential(
            nn.Linear(4 * 2 * hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def encode(self, ids, lens):
       
        emb = self.embed(ids)
    
        lens_cpu = lens.cpu().clamp(min=1)
        packed = nn.utils.rnn.pack_padded_sequence(emb, lens_cpu, batch_first=True,
                                                    enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        # Mean-pool over actual length
        mask = (ids != PAD).unsqueeze(-1).float()
        pooled = (out * mask[:, :out.size(1), :]).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.dropout(pooled)

    def forward(self, q_ids, q_len, opt_ids, opt_lens):
        # q_ids: (B, L), opt_ids: (B, 5, L)
        B = q_ids.size(0)
        h_q = self.encode(q_ids, q_len)                   
    
        opt_ids_flat = opt_ids.view(B*5, -1)
        opt_lens_flat = opt_lens.view(B*5)
        h_o = self.encode(opt_ids_flat, opt_lens_flat)  
        h_o = h_o.view(B, 5, -1)
        h_q_exp = h_q.unsqueeze(1).expand(-1, 5, -1)  
        diff = torch.abs(h_q_exp - h_o)
        prod = h_q_exp * h_o
        joined = torch.cat([h_q_exp, h_o, diff, prod], dim=-1)
        return self.head(joined).squeeze(-1)            

In [ ]:
def pairwise_margin_loss(scores, correct, margin=0.5):
  
    B = scores.size(0)
    pos = scores[torch.arange(B), correct].unsqueeze(1).expand(-1, 5)
    target = torch.ones_like(scores)
    # mask out the correct vs correct pair
    loss = F.margin_ranking_loss(pos, scores, target, margin=margin, reduction='none')
    mask = torch.ones_like(loss); mask[torch.arange(B), correct] = 0
    return (loss * mask).sum() / mask.sum().clamp(min=1)